<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Фактуры</h2>

----

### Вариант 10 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Invoice в C#, который будет представлять информацию о
фактурах за поставленные товары или оказанные услуги. На основе этого класса
разработать 2-3 производных класса, демонстрирующих принципы наследования и
полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и
методы, а также переопределены некоторые методы базового класса для
демонстрации полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [5]:
public class LineItem
{
    private string _name;
    private decimal _price;

    public string Name
    {
        get => _name;
        set => _name = string.IsNullOrWhiteSpace(value) ? "Без названия" : value;
    }

    public decimal Price
    {
        get => _price;
        set => _price = value < 0 ? 0 : value;
    }

    public LineItem()
    {
        Name = "Новая позиция";
        Price = 0;
    }

    public LineItem(string name, decimal price)
    {
        Name = name;   
        Price = price; 
    }
}
public class Invoice
{
    private string _invoiceNumber;
    private DateTime _issueDate;
    private decimal _totalAmount;
    public string InvoiceNumber
    {
        get => _invoiceNumber;
        set => _invoiceNumber = string.IsNullOrWhiteSpace(value) ? "INV-000" : value;
    }

    public DateTime IssueDate
    {
        get => _issueDate;
        set => _issueDate = value;
    }

    public decimal TotalAmount
    {
        get => _totalAmount;
        protected set => _totalAmount = value < 0 ? 0 : value;
    }

    public List<LineItem> Items { get; protected set; } = new List<LineItem>(); 
    public string ClientName { get; set; } = "Частное лицо"; 
    public bool IsPaid { get; protected set; } = false;      
    public decimal DiscountPercent { get; set; } = 0;       

    public Invoice()
    {
        InvoiceNumber = "INV-000";
        IssueDate = DateTime.Now;
    }

    public Invoice(string number, DateTime date) : this()
    {
        InvoiceNumber = number; 
        IssueDate = date;       
    }

    public Invoice(string number, DateTime date, string clientName) : this(number, date)
    {
        ClientName = clientName;
    }

    public virtual decimal CalculateTotal()
    {
        decimal baseTotal = Items.Sum(item => item.Price);
        TotalAmount = baseTotal - (baseTotal * (DiscountPercent / 100m));
        return TotalAmount;
    }

    public virtual void AddLine(LineItem lineItem)
    {
        if (lineItem == null) return;
        Items.Add(lineItem);
        Console.WriteLine($"[Счёт {InvoiceNumber}] Добавлена позиция: {lineItem.Name} — {lineItem.Price} руб.");
    }

    public void AddLine(string name, decimal price)
    {
        AddLine(new LineItem(name, price));
    }

    public virtual void RemoveLine(LineItem lineItem)
    {
        if (lineItem != null && Items.Remove(lineItem))
        {
            Console.WriteLine($"[Счёт {InvoiceNumber}] Удалена позиция: {lineItem.Name}");
        }
    }

    public void Pay() 
    {
        IsPaid = true;
        Console.WriteLine($"[Счёт {InvoiceNumber}] Полностью оплачен.");
    }

    public void Pay(string paymentMethod)
    {
        IsPaid = true;
        Console.WriteLine($"[Счёт {InvoiceNumber}] Оплачен способом: {paymentMethod}.");
    }

    public virtual void PrintSummary()
    {
        Console.WriteLine($"\n--- Счёт {InvoiceNumber} ---");
        Console.WriteLine($"Клиент: {ClientName} | Дата: {IssueDate.ToShortDateString()}");
        Console.WriteLine($"Статус: {(IsPaid ? "Оплачен" : "Ожидает оплаты")} | Скидка: {DiscountPercent}%");
        Console.WriteLine($"Позиций: {Items.Count} | Итоговая сумма: {CalculateTotal()} руб.");
    }

    // Взаимодействия
    public void SendItemTo(LineItem item, Invoice target)
    {
        if (item == null || target == null || target == this) return;

        if (Items.Contains(item))
        {
            RemoveLine(item);
            target.AddLine(item);
            Console.WriteLine($"--> [Взаимодействие] Позиция '{item.Name}' перенесена из '{InvoiceNumber}' в '{target.InvoiceNumber}'");
        }
    }

    public void MergeWith(Invoice other)
    {
        if (other == null || other == this) return;

        Console.WriteLine($"\n--> [Взаимодействие] Слияние счёта '{other.InvoiceNumber}' в счёт '{this.InvoiceNumber}'...");
        List<LineItem> toMove = new List<LineItem>(other.Items);
        foreach (LineItem line in toMove)
        {
            other.SendItemTo(line, this);
        }
    }

    public void CompareTotalWith(Invoice other)
    {
        if (other == null) return;

        decimal myTotal = this.CalculateTotal();
        decimal otherTotal = other.CalculateTotal();

        Console.WriteLine($"\n--> [Взаимодействие] Сравнение счетов '{this.InvoiceNumber}' ({myTotal} руб.) и '{other.InvoiceNumber}' ({otherTotal} руб.):");
        if (myTotal > otherTotal)
            Console.WriteLine($"    Счёт '{this.InvoiceNumber}' больше на {myTotal - otherTotal} руб.");
        else if (myTotal < otherTotal)
            Console.WriteLine($"    Счёт '{other.InvoiceNumber}' больше на {otherTotal - myTotal} руб.");
        else
            Console.WriteLine("    Суммы счетов равны.");
    }
}

public class GoodsInvoice : Invoice
{
    private DateTime _supplyDate;

    public DateTime SupplyDate { get => _supplyDate; set => _supplyDate = value; }
    public string DeliveryAddress { get; set; } = "Самовывоз"; 
    public decimal ShippingCost { get; set; } = 0;            

    public GoodsInvoice(string number, DateTime date, DateTime supplyDate, string address = "Самовывоз") 
        : base(number, date)
    {
        SupplyDate = supplyDate;
        DeliveryAddress = address;
    }

    public override decimal CalculateTotal()
    {
        return base.CalculateTotal() + ShippingCost;
    }

    public override void AddLine(LineItem lineItem)
    {
        base.AddLine(lineItem);
        Console.WriteLine("    --> [Товар] Дата поставки: " + SupplyDate.ToShortDateString());
    }

    public override void PrintSummary()
    {
        base.PrintSummary();
        Console.WriteLine($"    [Товарный счёт] Адрес доставки: {DeliveryAddress} | Стоимость доставки: {ShippingCost} руб.");
    }
}

public class ServiceInvoice : Invoice
{
    private DateTime _serviceDate;

    public DateTime ServiceDate { get => _serviceDate; set => _serviceDate = value; }
    public string SpecialistName { get; set; } = "Не назначен"; 
    public int WarrantyDays { get; set; } = 30;                 

    public ServiceInvoice(string number, DateTime date, DateTime serviceDate, string specialist = "Не назначен") 
        : base(number, date)
    {
        ServiceDate = serviceDate;
        SpecialistName = specialist;
    }

    public override void RemoveLine(LineItem lineItem)
    {
        base.RemoveLine(lineItem);
        Console.WriteLine("    --> [Услуга] Причина удаления: Отмена услуги клиентом.");
    }

    public override void PrintSummary()
    {
        base.PrintSummary();
        Console.WriteLine($"    [Счёт услуг] Специалист: {SpecialistName} | Гарантия: {WarrantyDays} дней");
    }
}

public class CombinedInvoice : Invoice
{
    private bool _returnAllowed;
    public bool ReturnAllowed { get => _returnAllowed; set => _returnAllowed = value; }
    public string SpecialNotes { get; set; } = "Нет примечаний"; 

    public CombinedInvoice(string number, DateTime date, bool returnAllowed) : base(number, date)
    {
        ReturnAllowed = returnAllowed;
    }

    public override decimal CalculateTotal()
    {
        decimal total = base.CalculateTotal();
        if (ReturnAllowed)
        {
            total += total * 0.05m;
        }
        TotalAmount = total;
        return TotalAmount;
    }
    public override void PrintSummary()
    {
        base.PrintSummary();
        Console.WriteLine($"    [Комбинированный] Возврат разрешен: {(ReturnAllowed ? "Да" : "Нет")} | Примечания: {SpecialNotes}");
    }
}

public class InvoiceManager<T> where T : Invoice
{
    private List<T> _invoices = new List<T>();

    public void AddInvoice(T invoice)
    {
        _invoices.Add(invoice);
        Console.WriteLine($"[Generic Менеджер] Добавлен счёт №{invoice.InvoiceNumber}");
    }

    public void ProcessAllSummaries()
    {
        Console.WriteLine("\n=== [Generic Менеджер] Печать всех зарегистрированных счетов ===");
        foreach (var inv in _invoices)
        {
            inv.PrintSummary();
        }
    }

    public decimal GetGrandTotal()
    {
        return _invoices.Sum(inv => inv.CalculateTotal());
    }
}

Console.WriteLine("=== 1. СОЗДАНИЕ ОБЪЕКТОВ И ДЕМОНСТРАЦИЯ ПЕРЕГРУЗОК (OVERLOADING) ===");

LineItem laptop = new LineItem("Ноутбук", 50000);
LineItem mouse = new LineItem("Мышь", 1200);

GoodsInvoice goodsInv = new GoodsInvoice("Т-001", DateTime.Now, DateTime.Now.AddDays(3), "ул. Ленина, д. 10");
goodsInv.ShippingCost = 500;
goodsInv.ClientName = "ООО Технопром";

ServiceInvoice serviceInv = new ServiceInvoice("У-001", DateTime.Now, DateTime.Now.AddDays(1), "Иванов И.И.");
serviceInv.DiscountPercent = 10;

CombinedInvoice combinedInv = new CombinedInvoice("К-001", DateTime.Now, true);
combinedInv.SpecialNotes = "Срочная доставка и настройка";

goodsInv.AddLine(laptop);
goodsInv.AddLine(mouse);

serviceInv.AddLine("Консультация", 8000); 
serviceInv.AddLine("Настройка ПО", 2500);  

combinedInv.AddLine("Кабель HDMI", 1500);

goodsInv.Pay("Банковский перевод"); 
serviceInv.Pay();                  

Console.WriteLine("\n=== 2. ДЕМОНСТРАЦИЯ ВЗАИМОДЕЙСТВИЯ ОБЪЕКТОВ ===");

goodsInv.SendItemTo(mouse, serviceInv);
goodsInv.CompareTotalWith(serviceInv);
combinedInv.MergeWith(serviceInv);

Console.WriteLine("\n=== 3. ДЕМОНСТРАЦИЯ GENERIC КЛАССА И ПОЛИМОРФИЗМА (OVERRIDING) ===");

InvoiceManager<Invoice> manager = new InvoiceManager<Invoice>();
manager.AddInvoice(goodsInv);
manager.AddInvoice(serviceInv);
manager.AddInvoice(combinedInv);

manager.ProcessAllSummaries();

Console.WriteLine($"\n Общая сумма всех счетов в менеджере: {manager.GetGrandTotal()} руб.");

=== 1. СОЗДАНИЕ ОБЪЕКТОВ И ДЕМОНСТРАЦИЯ ПЕРЕГРУЗОК (OVERLOADING) ===
[Счёт Т-001] Добавлена позиция: Ноутбук — 50000 руб.
    --> [Товар] Дата поставки: 23.10.2026
[Счёт Т-001] Добавлена позиция: Мышь — 1200 руб.
    --> [Товар] Дата поставки: 23.10.2026
[Счёт У-001] Добавлена позиция: Консультация — 8000 руб.
[Счёт У-001] Добавлена позиция: Настройка ПО — 2500 руб.
[Счёт К-001] Добавлена позиция: Кабель HDMI — 1500 руб.
[Счёт Т-001] Оплачен способом: Банковский перевод.
[Счёт У-001] Полностью оплачен.

=== 2. ДЕМОНСТРАЦИЯ ВЗАИМОДЕЙСТВИЯ ОБЪЕКТОВ ===
[Счёт Т-001] Удалена позиция: Мышь
[Счёт У-001] Добавлена позиция: Мышь — 1200 руб.
--> [Взаимодействие] Позиция 'Мышь' перенесена из 'Т-001' в 'У-001'

--> [Взаимодействие] Сравнение счетов 'Т-001' (50500 руб.) и 'У-001' (10530,0 руб.):
    Счёт 'Т-001' больше на 39970,0 руб.

--> [Взаимодействие] Слияние счёта 'У-001' в счёт 'К-001'...
[Счёт У-001] Удалена позиция: Консультация
    --> [Услуга] Причина удаления: Отмена услуги клиентом.
[